# Potato Leaf Disease Detection with YOLOv4
## Stage 1: Environment Setup & Darknet Build
Based on Aldakheel, Zakariah, and Alabdalall (2024), adapted for 3 potato classes.

In [ ]:
# Cell 1: Check GPU availability, GPU model, and compute capability
!nvidia-smi

import torch
if not torch.cuda.is_available():
    raise SystemError("GPU device not found! In Colab, navigate to: Runtime > Change runtime type > T4 GPU.")

gpu_name = torch.cuda.get_device_name(0)
compute_cap = torch.cuda.get_device_capability(0)
print(f"Detected GPU: {gpu_name}")
print(f"Compute Capability: {compute_cap[0]}.{compute_cap[1]}")
print("GPU verification successful.")

In [ ]:
# Cell 2: Mount Google Drive and initialize project folder structure
from google.colab import drive
import os

drive.mount('/content/drive')

PROJECT_ROOT = "/content/drive/MyDrive/potato-yolov4"
subdirs = [
    "data/raw",
    "data/processed/train",
    "data/processed/valid",
    "data/processed/test",
    "data/manifests",
    "weights/backup",
    "scripts",
    "results",
    "cfg"
]

for d in subdirs:
    os.makedirs(os.path.join(PROJECT_ROOT, d), exist_ok=True)

print("Created project directory tree under:", PROJECT_ROOT)
for d in subdirs:
    p = os.path.join(PROJECT_ROOT, d)
    assert os.path.exists(p), f"Directory creation failed: {p}"
print("All project directories successfully verified.")

In [ ]:
# Cell 3: Clone AlexeyAB Darknet repository to local Colab instance storage
import os

%cd /content
if not os.path.exists("/content/darknet"):
    !git clone https://github.com/AlexeyAB/darknet.git
else:
    print("Darknet directory already exists in /content/darknet.")

%cd /content/darknet
assert os.path.exists("/content/darknet/Makefile"), "Makefile not found! Git clone failed."
print("Cloned AlexeyAB/darknet successfully into local NVMe storage.")

In [ ]:
# Cell 4: Configure Makefile with sed for GPU, CUDNN, OpenCV, Libso, and Tesla T4 ARCH
%cd /content/darknet

# Enable hardware acceleration and library generation
!sed -i 's/GPU=0/GPU=1/' Makefile
!sed -i 's/CUDNN=0/CUDNN=1/' Makefile
!sed -i 's/CUDNN_HALF=0/CUDNN_HALF=1/' Makefile
!sed -i 's/OPENCV=0/OPENCV=1/' Makefile
!sed -i 's/LIBSO=0/LIBSO=1/' Makefile

# Configure sm_75 architecture for Tesla T4
!sed -i 's/^ARCH= -gencode arch=compute_35,code=sm_35/#ARCH= -gencode arch=compute_35,code=sm_35/' Makefile
!sed -i 's/^ARCH= -gencode arch=compute_50,code=sm_50/#ARCH= -gencode arch=compute_50,code=sm_50/' Makefile
!sed -i 's/^# ARCH= -gencode arch=compute_75,code=sm_75/ARCH= -gencode arch=compute_75,code=sm_75/' Makefile

print("--- Configured Makefile flags ---")
!head -n 10 Makefile | grep -E "GPU=|CUDNN=|CUDNN_HALF=|OPENCV=|LIBSO="
print("\n--- Active Architecture line ---")
!grep -E "arch=compute_75" Makefile

In [ ]:
# Cell 5: Compile Darknet binary and confirm executable usage
import os
%cd /content/darknet
!make -j$(nproc)

assert os.path.isfile("/content/darknet/darknet"), "Darknet build failed; executable not generated!"
print("\nTesting ./darknet execution:")
!./darknet

In [ ]:
# Cell 6: Download pre-trained backbone weights yolov4.conv.137
import os

WEIGHTS_URL = "https://github.com/AlexeyAB/darknet/releases/download/darknet_yolo_v3_optimal/yolov4.conv.137"
LOCAL_WEIGHTS = "/content/darknet/yolov4.conv.137"
DRIVE_WEIGHTS = "/content/drive/MyDrive/potato-yolov4/weights/yolov4.conv.137"

if not os.path.exists(LOCAL_WEIGHTS) and not os.path.exists(DRIVE_WEIGHTS):
    print(f"Downloading weights from official source:\n{WEIGHTS_URL}")
    !wget --quiet --show-progress {WEIGHTS_URL} -O {LOCAL_WEIGHTS}
    !cp {LOCAL_WEIGHTS} {DRIVE_WEIGHTS}
elif os.path.exists(DRIVE_WEIGHTS) and not os.path.exists(LOCAL_WEIGHTS):
    print("Copying weights from Google Drive backup...")
    !cp {DRIVE_WEIGHTS} {LOCAL_WEIGHTS}

file_size = os.path.getsize(LOCAL_WEIGHTS)
print(f"Weights path: {LOCAL_WEIGHTS}")
print(f"Size: {file_size:,} bytes ({file_size / (1024*1024):.2f} MB)")

# Official expected size is 170,007,816 bytes (~162.1 MB)
assert abs(file_size - 170007816) < 1000, f"Size mismatch! Got {file_size} bytes. File may be corrupted or truncated."
print("Weights size check PASSED.")

In [ ]:
# Cell 7: End-to-end verification summary of Stage 1
import os, torch

checks = {
    "GPU Available (Tesla T4)": torch.cuda.is_available() and "T4" in torch.cuda.get_device_name(0),
    "Google Drive Mounted": os.path.exists("/content/drive/MyDrive/potato-yolov4"),
    "Darknet Binary Executable": os.path.isfile("/content/darknet/darknet") and os.access("/content/darknet/darknet", os.X_OK),
    "Darknet Shared Lib (.so)": os.path.isfile("/content/darknet/libdarknet.so"),
    "yolov4.conv.137 Local": os.path.isfile("/content/darknet/yolov4.conv.137") and os.path.getsize("/content/darknet/yolov4.conv.137") > 160000000,
    "yolov4.conv.137 in Drive": os.path.isfile("/content/drive/MyDrive/potato-yolov4/weights/yolov4.conv.137")
}

print(f"{'CHECK ITEM':<32} | {'STATUS':<8}")
print("-" * 45)
all_ok = True
for k, v in checks.items():
    status = "OK" if v else "FAILED"
    if not v:
        all_ok = False
    print(f"{k:<32} | {status:<8}")

assert all_ok, "One or more Stage 1 checks failed! Check the output above."
print("\nSTAGE 1 COMPLETE: All components ready for Stage 2.")